In [0]:
# Inspect the existing Silver table
from pyspark.sql import functions as F
from delta.tables import DeltaTable

catalog = spark.sql(
    "SELECT current_catalog()"
).first()[0]

schema_name = "retail_lakehouse"

silver_table = f"{catalog}.{schema_name}.silver_orders"
rejected_table = f"{catalog}.{schema_name}.rejected_orders"

silver = spark.table(silver_table)

print("Existing Silver records:", silver.count())
display(silver.orderBy("order_id"))


In [0]:
# Simulate a new incoming batch
incoming_data = [
    # A new, valid order
    ("O1011", "2026-10-06", "C006", "P101",
     2, 499.0, "Completed", "West"),

    # The same order delivered twice
    ("O1011", "2026-10-06", "C006", "P101",
     2, 499.0, "Completed", "West"),

    # An invalid order: negative quantity
    ("O1012", "2026-10-06", "C007", "P102",
     -2, 999.0, "Completed", "South"),

    # Another valid order
    ("O1013", "2026-10-07", "C008", "P103",
     1, 199.0, "Completed", "North")
]

columns = [
    "order_id", "order_date", "customer_id", "product_id",
    "quantity", "unit_price", "status", "region"
]

incoming_df = spark.createDataFrame(incoming_data, columns)

incoming_df = incoming_df.withColumn(
    "order_date",
    F.to_date("order_date", "yyyy-MM-dd")
)

print("Incoming records:", incoming_df.count())
display(incoming_df)

In [0]:
# Deduplicate and split valid and rejected records
# Remove repeated deliveries within this incoming batch.
batch_deduplicated = incoming_df.dropDuplicates(["order_id"])

valid_condition = (
    F.col("order_id").isNotNull() &
    F.col("order_date").isNotNull() &
    F.col("product_id").isNotNull() &
    F.col("quantity").isNotNull() &
    (F.col("quantity") > 0) &
    F.col("unit_price").isNotNull() &
    (F.col("unit_price") >= 0)
)

valid_batch = batch_deduplicated.filter(valid_condition)
rejected_batch = batch_deduplicated.filter(~valid_condition)

print("Unique incoming orders:", batch_deduplicated.count())
print("Valid incoming orders:", valid_batch.count())
print("Rejected incoming orders:", rejected_batch.count())

display(valid_batch)
display(rejected_batch)

In [0]:
# Merge valid orders into Silver
# Insert only new order IDs
# Record the baseline before the merge
before_count = spark.table(silver_table).count()

silver_delta = DeltaTable.forName(spark, silver_table)

(
    silver_delta.alias("target")
    .merge(
        valid_batch.alias("source"),
        "target.order_id = source.order_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

after_count = spark.table(silver_table).count()

print("Silver rows before:", before_count)
print("Silver rows after:", after_count)
print("New rows inserted:", after_count - before_count)

In [0]:
# Save rejected records safel
# Merge invalid records into the rejection table
rejected_delta = DeltaTable.forName(spark, rejected_table)

(
    rejected_delta.alias("target")
    .merge(
        rejected_batch.alias("source"),
        "target.order_id = source.order_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Rejected records saved.")

In [0]:
# Validate incremental processing
# Verify uniqueness and persistence

final_silver = spark.table(silver_table)
final_rejected = spark.table(rejected_table)

duplicate_count = (
    final_silver
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicate_count == 0
assert final_silver.filter(
    F.col("order_id") == "O1011"
).count() == 1

assert final_silver.filter(
    F.col("order_id") == "O1013"
).count() == 1

assert final_rejected.filter(
    F.col("order_id") == "O1012"
).count() == 1

print("Incremental pipeline validation passed.")
print("Silver rows:", final_silver.count())
print("Rejected rows:", final_rejected.count())
print("Duplicate order IDs:", duplicate_count)

display(final_silver.orderBy("order_id"))
display(final_rejected.orderBy("order_id"))

In [0]:
# Test idempotency
# # Run the same batch again
